# 03 — Word & Sentence Embeddings

**Day 2 | Data Engineering & AI Bootcamp**

Embeddings convert text into numbers that machines can process.
The magic: words and sentences with similar **meaning** end up close together in vector space.
Every modern AI system — chatbots, search engines, recommendation systems — uses embeddings.

> **Note:** The first run downloads the `all-MiniLM-L6-v2` model (~90 MB). Subsequent runs are instant.

In [ ]:
import sys
sys.path.insert(0, '../src')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

from day2.embeddings import (
    MODEL_CATALOGUE, model_selection_guide,
    EmbeddingEncoder, dense_vs_sparse_comparison,
)

sns.set_theme(style='whitegrid')
print('Setup complete ✓')

## 1. Embedding Model Catalogue

Not all embedding models are equal. They differ in:
- **Dimensions:** 384 (fast/small) → 1536 (OpenAI, best quality)
- **Speed:** local models run on CPU in milliseconds; API models add network latency
- **Languages:** English-only vs multilingual (50+ languages)
- **Context length:** how many tokens of input the model can handle

> **Rule of thumb:** Start with `all-MiniLM-L6-v2` (384 dims, fast, free).
> Upgrade to `BAAI/bge-large-en-v1.5` (1024 dims) if quality is not enough.
> Use `text-embedding-3-small` (OpenAI, 1536 dims) for production-grade RAG.

In [ ]:
catalogue_df = pd.DataFrame([
    {
        'Model':          m.name.split('/')[-1],
        'Dims':           m.dims,
        'Max Tokens':     m.max_tokens,
        'Speed':          m.speed,
        'Quality':        m.quality,
        'Size (MB)':      m.size_mb,
        'Multilingual':   '✓' if m.multilingual else '',
        'Notes':          m.notes[:60],
    }
    for m in MODEL_CATALOGUE
])

display(catalogue_df)

# Model selection guide
print('\n--- Model Selection ---')
print(f"Speed priority (free):     {model_selection_guide(speed_priority=True).name}")
print(f"Quality priority (free):   {model_selection_guide(speed_priority=False).name}")
print(f"Multilingual:              {model_selection_guide(multilingual=True).name}")

## 2. Dense vs Sparse Vectors

**Sparse vectors** (TF-IDF, BM25): high-dimensional (vocab size = 50,000+), mostly zeros.
They capture **exact keyword presence**. Fast to compute, interpretable.
Weakness: 'dog' and 'canine' are completely unrelated in sparse space.

**Dense vectors** (sentence-transformers, BERT, OpenAI): low-dimensional (384–3072), all non-zero.
They capture **semantic meaning**. Slower to compute, require a trained model.
Strength: 'dog' and 'canine' are close in vector space.

> **Production best practice:** Use **hybrid search** — combine both.
> Weaviate and Azure AI Search do this natively. It beats pure semantic or pure keyword search.

In [ ]:
corpus = [
    'The dog ran across the grassy field.',
    'A canine sprinted through the meadow.',       # same meaning, zero word overlap with above
    'Machine learning models predict outcomes.',
    'Python is popular for data science projects.',
    'Deep neural networks learn representations.',
    'Vector databases store embedding vectors.',
]

r = dense_vs_sparse_comparison(corpus)
print(f'TF-IDF matrix shape:   {r["sparse_shape"]}')
print(f'TF-IDF sparsity:       {r["sparse_sparsity"]:.1%} zeros')
print(f'TF-IDF non-zero/doc:   {r["sparse_nonzero_per_doc"]} terms on average')
print(f'Dense shape:           {r["dense_shape"]}')
print(f'Dense sparsity:        {r["dense_sparsity"]}')
print(f'\nThe semantic gap:')
for k, v in r['semantic_gap_example'].items():
    print(f'  {v}')

## 3. Generating Embeddings with sentence-transformers

`sentence-transformers` is the standard library for local embedding generation.
It wraps Hugging Face models and makes encoding as simple as `model.encode(texts)`.
Outputs are L2-normalised by default — each vector has magnitude 1.0.

> **Real world:** In a Databricks RAG pipeline, you call `model.encode()` on document chunks
> during the indexing phase, then again on the user query at retrieval time.
> The same model must be used for both — otherwise the vector spaces won't match.

In [ ]:
# Load model (downloads ~90MB on first run)
encoder = EmbeddingEncoder('all-MiniLM-L6-v2')
print(f'Model: {encoder.model_name}')
print(f'Embedding dimensions: {encoder.dims}')

# Encode sample texts
texts = [
    'Machine learning enables intelligent systems.',
    'Deep learning uses many-layered neural networks.',
    'Natural language processing understands human text.',
    'Data pipelines move and transform data at scale.',
    'Vector databases support fast similarity search.',
]

vecs = encoder.encode(texts, normalize=True)
print(f'\nOutput shape:  {vecs.shape}')
print(f'First vector (first 8 dims): {vecs[0, :8].round(4)}')
print(f'Vector norms (should all be 1.0): {np.linalg.norm(vecs, axis=1).round(5)}')

## 4. OpenAI Embedding API — Text → Vector in One Call

OpenAI provides hosted embedding models via a simple REST API.
No model download needed — you send text, you get a vector back.
This is how most **production RAG systems** generate embeddings when quality is critical.

```
pip install openai
export OPENAI_API_KEY='sk-...'
```

> **`text-embedding-3-small`** (1536 dims) — best cost/quality ratio.  
> **`text-embedding-3-large`** (3072 dims) — highest quality, higher cost.  
> Both support **Matryoshka** — you can shrink dimensions (e.g. 256) with no re-training.

In [ ]:
# ── OpenAI Embeddings — the simplest possible example ─────────────────────
# Requires: pip install openai
# Set your key: export OPENAI_API_KEY='sk-...'

import os
import numpy as np

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY', '')  # reads from env var

if OPENAI_API_KEY:
    from openai import OpenAI

    client = OpenAI()  # auto-reads OPENAI_API_KEY from env

    # ── Step 1: encode a single sentence ──────────────────────────────────
    text = 'Machine learning transforms raw data into intelligent predictions.'

    response = client.embeddings.create(
        model = 'text-embedding-3-small',  # 1536 dims
        input = text,
    )

    vector = response.data[0].embedding   # plain Python list of floats

    print(f'Text:       {text}')
    print(f'Model:      text-embedding-3-small')
    print(f'Dimensions: {len(vector)}')
    print(f'First 8:    {[round(v, 5) for v in vector[:8]]}')
    print(f'Norm:       {np.linalg.norm(vector):.6f}  (OpenAI returns unit vectors)')

    # ── Step 2: batch encode multiple texts ───────────────────────────────
    texts = [
        'The cat sat on the mat.',
        'A feline rested on the rug.',      # same meaning, different words
        'Apache Spark is a distributed engine.',
    ]

    batch_resp = client.embeddings.create(
        model = 'text-embedding-3-small',
        input = texts,                         # pass a list — one API call
    )

    vecs = np.array([d.embedding for d in batch_resp.data])
    sim  = vecs @ vecs.T                       # cosine sim (unit vectors)

    print(f'\nBatch shape: {vecs.shape}')
    print(f'Similarity matrix:')
    print(np.round(sim, 3))
    print(f'\n"cat" ↔ "feline":  {sim[0,1]:.4f}  (high — same meaning)')
    print(f'"cat" ↔ "Spark":   {sim[0,2]:.4f}  (low  — unrelated topics)')

    # ── Step 3: Matryoshka — shrink dimensions without retraining ─────────
    compact_resp = client.embeddings.create(
        model      = 'text-embedding-3-small',
        input      = text,
        dimensions = 256,           # truncate to 256 — still valid unit vector
    )
    compact_vec = compact_resp.data[0].embedding
    print(f'\nMatryoshka 256-dim norm: {np.linalg.norm(compact_vec):.6f}')
    print(f'256 dims uses {256/1536:.0%} of storage vs full 1536-dim vector')

else:
    print('OPENAI_API_KEY not set — showing what the output looks like:\n')
    print('Text:       Machine learning transforms raw data into intelligent predictions.')
    print('Model:      text-embedding-3-small')
    print('Dimensions: 1536')
    print('First 8:    [0.02135, -0.04821, 0.01392, -0.00874, 0.03156, ...]')
    print('Norm:       1.000000  (OpenAI returns unit vectors)')
    print()
    print('Similarity matrix (cat/feline/Spark):')
    print('[[1.000  0.891  0.127]')
    print(' [0.891  1.000  0.131]')
    print(' [0.127  0.131  1.000]]')
    print()
    print('"cat" ↔ "feline":  0.8910  (high — same meaning)')
    print('"cat" ↔ "Spark":   0.1270  (low  — unrelated topics)')
    print()
    print('Set OPENAI_API_KEY in .env to run live ↑')


In [ ]:
# ── API call anatomy explained ─────────────────────────────────────────────
print('OpenAI Embeddings API — anatomy of a call:')
print()
print("  client.embeddings.create(")
print("      model  = 'text-embedding-3-small',   # which model")
print("      input  = 'any string, or list',       # your text")
print("      dimensions = 256,                     # optional: shrink (Matryoshka)")
print("  )")
print()
print('  response.data[0].embedding    → list[float]  (your vector)')
print('  response.usage.total_tokens   → int         (billed tokens)')
print()

# ── Compare: local vs OpenAI ───────────────────────────────────────────────
comparison = [
    {'Approach': 'sentence-transformers (local)',   'Model': 'all-MiniLM-L6-v2',       'Dims': 384,  'Cost': 'Free', 'Latency': '~5ms',   'Privacy': 'Data stays local'},
    {'Approach': 'sentence-transformers (local)',   'Model': 'BAAI/bge-large-en-v1.5', 'Dims': 1024, 'Cost': 'Free', 'Latency': '~20ms',  'Privacy': 'Data stays local'},
    {'Approach': 'OpenAI API',                      'Model': 'text-embedding-3-small', 'Dims': 1536, 'Cost': '$0.02/1M tokens', 'Latency': '~100ms', 'Privacy': 'Sent to OpenAI'},
    {'Approach': 'OpenAI API',                      'Model': 'text-embedding-3-large', 'Dims': 3072, 'Cost': '$0.13/1M tokens', 'Latency': '~150ms', 'Privacy': 'Sent to OpenAI'},
]
display(pd.DataFrame(comparison))

## 5. Semantic Similarity Heat Map

Once texts are embedded, similarity is just a dot product (cosine similarity for unit vectors).
A similarity matrix shows how related every pair of sentences is.
You expect semantically related sentences to have high values (warm colours).

> **Real world:** This matrix is what powers 'find similar documents' in a RAG system.
> The user query is one vector; the matrix shows which documents are closest.
> Databricks Vector Search does this at billion-document scale using HNSW indexing.

In [ ]:
# Compute pairwise cosine similarity
sim_matrix = vecs @ vecs.T  # (5, 5) — dot product of unit vectors = cosine

short_labels = [t[:35] + '...' if len(t) > 35 else t for t in texts]

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    sim_matrix, annot=True, fmt='.3f', cmap='YlOrRd',
    vmin=0, vmax=1, linewidths=0.5, linecolor='white',
    xticklabels=short_labels, yticklabels=short_labels, ax=ax
)
ax.set_title('Cosine Similarity Matrix — Sentence Embeddings', fontweight='bold', fontsize=12)
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right', fontsize=8)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=8)
plt.tight_layout()
plt.show()

# Most and least similar pairs
np.fill_diagonal(sim_matrix, 0)
i, j = np.unravel_index(sim_matrix.argmax(), sim_matrix.shape)
k, l = np.unravel_index(sim_matrix.argmin(), sim_matrix.shape)
print(f'Most similar:  [{sim_matrix[i,j]:.3f}]  "{texts[i]}" ↔ "{texts[j]}"')
print(f'Least similar: [{sim_matrix[k,l]:.3f}]  "{texts[k]}" ↔ "{texts[l]}"')

## 6. Embedding Dimensions: 384 vs 768 vs 1024 vs 1536

Higher dimensions generally mean better quality — the model has more 'room' to store meaning.
But higher dimensions also mean: more memory, slower search, higher compute cost.
The right dimension depends on your quality threshold and infrastructure budget.

| Dims | Example Model              | Use Case                        |
|------|----------------------------|---------------------------------|
| 384  | all-MiniLM-L6-v2           | Dev, prototyping, low latency   |
| 768  | all-mpnet-base-v2, BERT    | Production English search       |
| 1024 | BAAI/bge-large-en-v1.5     | High-quality RAG, MTEB top tier |
| 1536 | text-embedding-3-small     | OpenAI API, long context        |
| 3072 | text-embedding-3-large     | OpenAI API, best quality        |

In [ ]:
dims_data = [
    {'Model': 'all-MiniLM-L6-v2',          'Dims': 384,  'Relative Quality': 72, 'Relative Speed': 100},
    {'Model': 'all-mpnet-base-v2',          'Dims': 768,  'Relative Quality': 80, 'Relative Speed': 55},
    {'Model': 'BAAI/bge-large-en-v1.5',     'Dims': 1024, 'Relative Quality': 91, 'Relative Speed': 25},
    {'Model': 'text-embedding-3-small',     'Dims': 1536, 'Relative Quality': 95, 'Relative Speed': 40},
    {'Model': 'text-embedding-3-large',     'Dims': 3072, 'Relative Quality': 99, 'Relative Speed': 20},
]
df_dims = pd.DataFrame(dims_data)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

colors = ['#2ecc71', '#3498db', '#9b59b6', '#e67e22', '#e74c3c']
ax1.barh(df_dims['Model'].str.split('/').str[-1], df_dims['Relative Quality'],
         color=colors, alpha=0.85, edgecolor='white')
ax1.set_xlabel('Relative Quality Score')
ax1.set_title('Quality vs Model', fontweight='bold')
ax1.set_xlim(60, 102)

ax2.scatter(df_dims['Dims'], df_dims['Relative Quality'],
            s=150, c=colors, alpha=0.9, edgecolors='white', linewidths=0.8, zorder=3)
for _, row in df_dims.iterrows():
    ax2.annotate(row['Model'].split('/')[-1][:18],
                 (row['Dims'], row['Relative Quality']),
                 xytext=(6, 0), textcoords='offset points', fontsize=8)
ax2.set_xlabel('Embedding Dimensions')
ax2.set_ylabel('Relative Quality')
ax2.set_title('Dimensions vs Quality', fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

display(df_dims)

## 7. Visualising the Vector Space (PCA → 2D)

Embeddings live in high-dimensional space (384+ dims) — impossible to visualise directly.
PCA (Principal Component Analysis) projects them to 2D by finding the axes of maximum variance.
Semantically similar sentences should cluster together, even in 2D.

> This is how you **validate** that your embedding model is working correctly.
> If topic clusters are visually separated, the model is capturing meaning.
> If everything is a random cloud, the model may be wrong for your domain.

In [ ]:
extended_corpus = [
    # ML cluster
    'Supervised learning requires labelled training data.',
    'Random forests ensemble many decision trees.',
    'Cross-validation estimates model generalisation.',
    'Gradient descent minimises the loss function.',
    # NLP cluster
    'Tokenisation splits text into sub-word units.',
    'Named entity recognition finds people and places.',
    'Sentiment analysis classifies text as positive or negative.',
    'Large language models are trained on trillions of tokens.',
    # Data Engineering cluster
    'ETL pipelines extract transform and load data.',
    'Apache Spark processes data across distributed clusters.',
    'Delta Lake adds ACID transactions to Parquet files.',
    'Batch jobs run on a scheduled interval to process data.',
]

group_labels = ['ML']*4 + ['NLP']*4 + ['Data Eng']*4
group_colors = {'ML': '#e74c3c', 'NLP': '#3498db', 'Data Eng': '#2ecc71'}

vecs_ext = encoder.encode(extended_corpus)

pca  = PCA(n_components=2, random_state=42)
pts  = pca.fit_transform(vecs_ext)
expl = pca.explained_variance_ratio_

fig, ax = plt.subplots(figsize=(10, 7))
for group in set(group_labels):
    idx = [i for i, g in enumerate(group_labels) if g == group]
    ax.scatter(pts[idx, 0], pts[idx, 1],
               label=group, color=group_colors[group],
               s=160, alpha=0.85, edgecolors='white', linewidths=0.8)

for i, text in enumerate(extended_corpus):
    short = text[:30]
    ax.annotate(short, (pts[i, 0], pts[i, 1]),
                fontsize=7.5, xytext=(4, 3), textcoords='offset points', alpha=0.9)

ax.set_xlabel(f'PC1 ({expl[0]:.1%} variance)', fontsize=11)
ax.set_ylabel(f'PC2 ({expl[1]:.1%} variance)', fontsize=11)
ax.set_title('Sentence Embeddings in 2D\n(Topic clusters visible — model captures meaning)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()